# 08 - EAD, CCF, LGD & Expected Loss (framework-aligned)

**Plain-English summary (read this first)**

When a borrower stops paying, the bank loses money on whatever they still owe at that moment - the
**EAD (Exposure at Default)**. This notebook measures EAD on the **credit-card** accounts in the data,
the related **CCF (Credit Conversion Factor)** (how much *unused* limit is drawn on the way into
default), and combines them with the scorecard PD and a benchmarked LGD into **Expected Loss = PD x LGD x EAD**.

This version is brought into line with the APS 113 / APG 113 / Basel CRE36 EAD framework:

- **EAD-3:** the **primary** exposure anchor is the **onset of delinquency** (the start of the late
  streak), not the balance at the late-firing 90+DPD flag (kept only as a secondary/diagnostic figure).
- **EAD-2:** EAD uses the **total receivable** (principal + accrued interest + fees), never capped at drawn balance.
- **EAD-1:** we **keep every account** (including fully-drawn / over-limit ones) and **do not clip the
  observed CCF**; in the high-utilisation *region of instability* we switch the CCF basis from the
  undrawn-limit factor (ULF) to a limit/balance factor (LF/BF).
- **EAD-4:** long-run (count-weighted) CCF, a downturn CCF, a margin of conservatism, the allowed
  homogeneity exclusion, and the EAD >= drawn-balance floor are all shown.
- **LGD-1 / EL-1:** LGD is a benchmarked **base/downturn range** (no internal recoveries), and the EL
  example is **account-level** - each card account linked to its own borrower PD - shown across the LGD range.

> **Important - read the "Data-quality finding" section near the end.** A sanity check shows the 90+DPD
> flag on this dataset **does not represent economic default** (`SK_DPD` accumulates and never resets).
> So the EAD/CCF/EL numbers here are a **methodology demonstration, not a credible loss estimate** - the
> methodology is now correct; the *data* cannot support a credible revolving CCF, which is itself the finding.

## What this notebook does, step by step

1. Read the credit-card monthly file, including the **receivable / excess** columns (EAD-2).
2. Mark an account **defaulted** at its first **90+DPD** month, then walk back to the **onset of
   delinquency** - the start of the late run (EAD-3). Onset is the primary EAD anchor.
3. **EAD** = total receivable at onset (>= drawn balance). **Reference point** = 12 months before onset.
4. **CCF** on three bases (ULF / LF / BF), keeping all accounts and not clipping (EAD-1).
5. Add the EAD-4 elements: homogeneity exclusion, long-run count-weighted CCF, downturn CCF, MoC, floor.
6. **LGD** as a benchmarked base/downturn range (LGD-1) and **EL = PD x LGD x EAD** account-level (EL-1).

**Columns used (real names in `data/raw/credit_card_balance.csv`):**

| Meaning | Column |
|---|---|
| Account id / Client id | `SK_ID_PREV` / `SK_ID_CURR` |
| Month index (-1 = freshest) | `MONTHS_BALANCE` |
| Balance owed / Credit limit | `AMT_BALANCE` / `AMT_CREDIT_LIMIT_ACTUAL` |
| **Total receivable (principal + interest + fees)** | `AMT_TOTAL_RECEIVABLE` |
| Receivable principal | `AMT_RECEIVABLE_PRINCIPAL` |
| Drawings / Days past due | `AMT_DRAWINGS_CURRENT` / `SK_DPD` |

In [1]:
# Load libraries, locate the repo root, and import the shared PD floor (PD-1) for the EL link.
import pandas as pd
import numpy as np
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "data" / "raw" / "credit_card_balance.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent          # walk up until we find the data folder
sys.path.insert(0, str(ROOT))
from src.calibration import apply_pd_floor      # PD-1: 5 bps floor, reused for the EL link

DATA = ROOT / "data" / "raw" / "credit_card_balance.csv"
SC   = ROOT / "outputs" / "tables" / "scorecard_outputs"
OUT_DIR = ROOT / "outputs" / "tables"; OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", ROOT)

Repo root: D:\Jane\Job Search\Github\bank\github project\scorecard pd ead consummer credit


In [2]:
# EAD-2: read the receivable/excess columns too, not just the drawn balance.
cols = ["SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE", "AMT_BALANCE",
        "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_CURRENT",
        "AMT_TOTAL_RECEIVABLE", "AMT_RECEIVABLE_PRINCIPAL", "SK_DPD"]
cc = pd.read_csv(DATA, usecols=cols)
print(f"{len(cc):,} monthly rows across {cc.SK_ID_PREV.nunique():,} credit-card accounts")
cc.head()

3,840,312 monthly rows across 104,307 credit-card accounts


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_TOTAL_RECEIVABLE,SK_DPD
0,2562384,378907,-6,56.970,135000,877.5,0.000,0.000,0
1,2582071,363914,-1,63975.555,45000,2250.0,60175.080,64875.555,0
2,1740877,371185,-7,31815.225,450000,0.0,26926.425,31460.085,0
3,1389973,337855,-4,236572.110,225000,2250.0,224949.285,233048.970,0
4,1891521,126868,-1,453919.455,450000,11547.0,443044.395,453919.455,0


In [3]:
# Default month = first (earliest) month an account is 90+ days past due.
dmonth = (cc[cc.SK_DPD >= 90].groupby("SK_ID_PREV")["MONTHS_BALANCE"].min()
          .rename("def_month").reset_index())
print(f"{len(dmonth):,} accounts ever reached 90+ days past due")

1,806 accounts ever reached 90+ days past due


## EAD-3 - Re-anchor exposure on the onset of delinquency

The 90+DPD flag on this data fires **late**, after months of an accumulating days-past-due counter, by
which point the balance is often a tiny stranded residual. The closest this dataset gets to a true
"point of default" is the **onset of delinquency** - the first month of the late streak that ends in
the 90+DPD flag. We therefore make **EAD at onset the primary (headline) figure** and keep
**balance-at-90+DPD as a clearly-labelled secondary/diagnostic** number. The CCF reference point is
then taken 12 months before **onset**.

In [4]:
# EAD-3: onset of delinquency = the START of the late streak that ends at the 90+DPD month.
sub = (cc.merge(dmonth, on="SK_ID_PREV").query("MONTHS_BALANCE <= def_month")
         .sort_values(["SK_ID_PREV", "MONTHS_BALANCE"]))

def _onset_month(g):
    dpd, m = g.SK_DPD.values, g.MONTHS_BALANCE.values
    i = len(g) - 1                       # last row in the window = the 90+DPD default month
    while i > 0 and dpd[i - 1] > 0:      # step back while the prior month is also past due
        i -= 1
    return pd.Series({"onset_month": m[i]})

onset = sub.groupby("SK_ID_PREV")[["MONTHS_BALANCE", "SK_DPD"]].apply(_onset_month).reset_index()
acc = dmonth.merge(onset, on="SK_ID_PREV")
print(f"onset-of-delinquency month computed for {len(acc):,} defaulted accounts")

onset-of-delinquency month computed for 1,806 defaulted accounts


In [5]:
# Receivables-inclusive exposure at any chosen month (EAD-2), with the EAD >= drawn-balance floor (EAD-4).
def exposure_at(month_col):
    key = acc[["SK_ID_PREV", month_col]].rename(columns={month_col: "_m"})
    j = cc.merge(key, on="SK_ID_PREV").query("MONTHS_BALANCE == _m")
    return (j.groupby("SK_ID_PREV", as_index=False)
              .agg(bal=("AMT_BALANCE", "max"), recv=("AMT_TOTAL_RECEIVABLE", "max"),
                   limit=("AMT_CREDIT_LIMIT_ACTUAL", "max"), dpd=("SK_DPD", "max")))

ons = exposure_at("onset_month").rename(columns={"bal":"bal_onset","recv":"recv_onset","limit":"lim_onset","dpd":"dpd_onset"})
dfl = exposure_at("def_month").rename(columns={"bal":"bal_90dpd","recv":"recv_90dpd","limit":"lim_90dpd","dpd":"dpd_90dpd"})
acc = acc.merge(ons, on="SK_ID_PREV").merge(dfl, on="SK_ID_PREV")

# EAD-2: total receivable (principal + accrued interest + fees), NOT capped at drawn balance.
# EAD-4 floor: EAD >= current drawn balance (Basel CRE36.89) -> enforced with np.maximum.
acc["EAD_onset"] = np.maximum(acc.recv_onset, acc.bal_onset)    # PRIMARY anchor (EAD-3)
acc["EAD_90dpd"] = np.maximum(acc.recv_90dpd, acc.bal_90dpd)    # secondary / diagnostic
print("PRIMARY   avg EAD at onset of delinquency:        {:,.0f}".format(acc.EAD_onset.mean()))
print("secondary avg EAD at 90+DPD (balance-at-flag):    {:,.0f}".format(acc.EAD_90dpd.mean()))
print("receivables exceed drawn balance in {:.1%} of accounts (accrued interest/fees now included)".format(
      (acc.recv_onset > acc.bal_onset + 1).mean()))

PRIMARY   avg EAD at onset of delinquency:        3,115
secondary avg EAD at 90+DPD (balance-at-flag):    4,521
receivables exceed drawn balance in 4.7% of accounts (accrued interest/fees now included)


## EAD-2 - what is (and is not) in EAD here

EAD is measured as **`AMT_TOTAL_RECEIVABLE`** (principal + accrued interest + other receivables), floored
at the drawn balance - so it is **not capped at principal/limit**, as APS 113 Att D EAD para 10 requires.

- **Included:** drawn principal, accrued interest and fees (via total receivable), and any limit excess
  that is reflected in the receivable balance.
- **Not separately available:** a clean, itemised "limit-excess" field and post-default drawings (the
  latter belong in LGD, not EAD - see EAD-4 note). Where a component cannot be isolated, EAD here should
  be read as a **lower bound** on true exposure.

In [6]:
# EAD-3: reference point = 12 months before ONSET. Record balance / limit / DPD there.
acc["ref_month"] = acc.onset_month - 12
ref = exposure_at("ref_month").rename(columns={"bal":"bal_ref","recv":"recv_ref","limit":"limit_ref","dpd":"dpd_ref"})
acc = acc.merge(ref, on="SK_ID_PREV")
acc["util_ref"]     = acc.bal_ref / acc.limit_ref.replace(0, np.nan)
acc["headroom_ref"] = acc.limit_ref - acc.bal_ref
print(f"{len(acc):,} accounts have a reference record 12 months before onset")

1,539 accounts have a reference record 12 months before onset


## EAD-1 - handle the region of instability properly (don't clip, don't drop)

Basel CRE36.95(2) / APG 113 para 129(c) name two **ineffective** mitigations that the old code used:
**capping observed CCFs at 0-100%** and **omitting** fully-drawn / over-limit accounts. Both are removed.

Instead we keep **every** account and compute three CCF bases:

- **ULF (undrawn-limit factor):** `(EAD - balance) / (limit - balance)` - the textbook CCF, but it
  **explodes when headroom -> 0** (the *region of instability*).
- **LF (limit factor):** `EAD / limit` - exposure as a fraction of the limit (stable when headroom is tiny).
- **BF (balance factor):** `EAD / balance` - exposure as a fraction of the drawn balance.

In the **high-utilisation** band (util > 0.80) ULF is unreliable, so we read **LF/BF** there
(Basel CRE36.95(1)). Utilisation is reported as a segment driver.

In [7]:
# EAD-1: keep ALL accounts (incl. fully-drawn / over-limit) and DO NOT clip observed CCF.
EAD = acc.EAD_onset
acc["ULF"] = np.where(acc.headroom_ref > 0, (EAD - acc.bal_ref) / acc.headroom_ref, np.nan)  # undrawn-limit factor
acc["LF"]  = np.where(acc.limit_ref > 0, EAD / acc.limit_ref, np.nan)                          # limit factor
acc["BF"]  = np.where(acc.bal_ref > 0,  EAD / acc.bal_ref,  np.nan)                            # balance factor

acc["util_band"] = pd.cut(acc.util_ref, [-np.inf, 0.30, 0.80, np.inf],
                          labels=["low (<=0.30)", "medium (0.30-0.80)", "high (>0.80)"])
ccf_by_band = (acc.groupby("util_band", observed=True)
                  .agg(accounts=("ULF", "size"), mean_ULF=("ULF", "mean"),
                       mean_LF=("LF", "mean"), mean_BF=("BF", "mean")).round(3))
print("CCF by utilisation band - no clip, no dropped accounts (EAD-1).")
print("In the high band ULF is unstable (headroom -> 0) -> read LF / BF instead:\n")
ccf_by_band

CCF by utilisation band - no clip, no dropped accounts (EAD-1).
In the high band ULF is unstable (headroom -> 0) -> read LF / BF instead:



,accounts,mean_ULF,mean_LF,mean_BF
util_band,,,,
low (<=0.30),390,-0.146,0.006,0.823
medium (0.30-0.80),758,-1.261,0.012,0.024
high (>0.80),388,-57.904,0.019,0.019


## EAD-4 - the remaining framework elements

- **Homogeneity exclusion (allowed):** APG 113 para 129(b) lets us drop accounts **already problematic
  at the reference date** (already delinquent 12m pre-default). This is the *allowed* exclusion - the
  opposite of the *forbidden* drop in EAD-1, which removed over-limit accounts from the healthy region.
- **Long-run, count-weighted CCF:** each default counts once (not exposure-weighted) - APS 113 Att D EAD para 2.
- **Downturn CCF:** a higher-percentile (stressed) CCF, confirmed **>= the long-run** average - APG 113 paras 131-132.
- **Margin of conservatism:** an add-on for thin data and **positive default/EAD correlation** (cards
  often draw harder into default) - Step 10 / APS 113 Att D EAD para 2.
- **Floor:** EAD **>= current drawn balance** - Basel CRE36.89 (already enforced via `np.maximum`).

In [8]:
# EAD-4 elements.
# (a) Homogeneity exclusion (ALLOWED): drop accounts already delinquent at the reference date.
clean = acc[acc.dpd_ref.fillna(0) == 0].copy()
print(f"Homogeneity exclusion: dropped {len(acc)-len(clean):,} accounts already delinquent at ref; kept {len(clean):,}")

# (b) Long-run, COUNT-weighted average CCF over the stable region (each default = one observation).
stable = clean[clean.util_ref <= 0.80]
lr_ccf  = float(stable.ULF.mean())
# (c) Downturn CCF: higher-percentile, confirmed >= long-run.
dt_ccf  = max(float(stable.ULF.quantile(0.75)), lr_ccf)
# (d) Margin of conservatism add-on (positive default/EAD correlation).
moc_ccf = 0.10 * abs(lr_ccf)
# (e) EAD floor check: EAD >= drawn balance.
floor_ok = float((acc.EAD_onset + 1 >= acc.bal_onset).mean())

print("Long-run count-weighted CCF (ULF, util<=0.80): {:.3f}".format(lr_ccf))
print("Downturn CCF (75th pct, >= long-run):          {:.3f}".format(dt_ccf))
print("MoC add-on (+10% of |long-run|):               {:.3f}".format(moc_ccf))
print("EAD floor satisfied (EAD >= drawn balance):    {:.1%} of accounts".format(floor_ok))

Homogeneity exclusion: dropped 163 accounts already delinquent at ref; kept 1,376
Long-run count-weighted CCF (ULF, util<=0.80): -0.883
Downturn CCF (75th pct, >= long-run):          -0.278
MoC add-on (+10% of |long-run|):               0.088
EAD floor satisfied (EAD >= drawn balance):    100.0% of accounts


**Reading the CCF numbers (the honest result).** On this book the long-run ULF comes out **negative**:
balances are paid **down** to a residual *before* default, so `EAD - balance_ref < 0`. A negative CCF has
no economic meaning - it is a direct symptom of the data-quality problem below, not a calculation error.
The methodology is now correct (all accounts kept, no clipping, basis switched in the unstable region),
but the **data cannot support a credible revolving CCF**. The operative EAD model therefore falls back to
the **floor**: `EAD ~= drawn balance at onset`.

**Post-default drawings -> LGD, not EAD.** Any drawing *after* the default point is treated in **LGD**,
not added to EAD (APS 113 Att D EAD para 1).

## LGD-1 - a benchmarked base/downturn range (not a single 0.70)

This dataset has **no recovery data**, so LGD cannot be measured internally - it is an **external-benchmark
assumption**. Rather than a single round number, we carry a **base** and a **downturn** figure inside a
published range for **unsecured consumer / credit-card** exposures (typically ~65-90%, downturn higher),
and we show EL across that whole range as a **sensitivity**. A real, recovery-based LGD is demonstrated in
the companion **Freddie Mac mortgage** project.

In [9]:
# LGD-1: external-benchmark base/downturn range (no internal recoveries on this book).
LGD_BASE, LGD_DOWNTURN = 0.75, 0.85
LGD_LOW,  LGD_HIGH     = 0.65, 0.90
lgd_grid = pd.DataFrame({
    "scenario": ["benchmark low", "base", "downturn", "benchmark high"],
    "LGD": [LGD_LOW, LGD_BASE, LGD_DOWNTURN, LGD_HIGH],
    "basis": ["lower end of published unsecured-consumer LGD",
              "central external benchmark (unsecured consumer / card)",
              "downturn / stressed LGD (higher, per Basel downturn-LGD principle)",
              "upper end of published unsecured-consumer LGD"],
})
print("LGD is ASSUMED from external benchmarks - a range is carried, not a single point:")
lgd_grid

LGD is ASSUMED from external benchmarks - a range is carried, not a single point:


,scenario,LGD,basis
0,benchmark low,0.65,lower end of published unsecured-consumer LGD
1,base,0.75,central external benchmark (unsecured consumer...
2,downturn,0.85,"downturn / stressed LGD (higher, per Basel dow..."
3,benchmark high,0.90,upper end of published unsecured-consumer LGD


## EL-1 - account-level Expected Loss across the LGD range

Instead of one portfolio-average PD, we link **each defaulted card account to its own borrower's
calibrated PD** (via `SK_ID_CURR`) and compute `EL = PD x LGD x EAD` per account, then show the total
across the LGD range. The scorecard's test split only covers part of the book, so link **coverage** is
reported and the example is **account-level but illustrative**.

In [10]:
# EL-1: link each card account to ITS borrower's calibrated PD, then EL across the LGD range.
scored = pd.read_csv(SC / "06_test_scored.csv", usecols=["SK_ID_CURR", "pd_pred"])
cs = pd.read_csv(SC / "calibration_summary.csv").iloc[0]
factor = float(cs.calibrated_mean_pd) / float(cs.raw_mean_pd)        # raw -> long-run calibrated
scored["pd_cal"] = apply_pd_floor(scored["pd_pred"] * factor)        # PD-1 floor applied

prev2curr = cc[["SK_ID_PREV", "SK_ID_CURR"]].drop_duplicates("SK_ID_PREV")   # each card account -> its client
linked = (clean.merge(prev2curr, on="SK_ID_PREV", how="left")
               .merge(scored[["SK_ID_CURR", "pd_cal"]], on="SK_ID_CURR", how="left"))
coverage = float(linked.pd_cal.notna().mean())
m = linked.pd_cal.notna()
print("Account-level link coverage: {:.1%} of clean card accounts matched a scored borrower".format(coverage))
print("(scorecard test split covers only part of the book -> account-level but illustrative)\n")

el_rows = []
for _, g in lgd_grid.iterrows():
    el = linked.pd_cal[m] * g.LGD * linked.EAD_onset[m]
    el_rows.append({"scenario": g.scenario, "LGD": g.LGD,
                    "mean_EL_per_account": round(float(el.mean()), 2),
                    "segment_EL_total": round(float(el.sum()), 0)})
el_sensitivity = pd.DataFrame(el_rows)
print("EL across the LGD range (account-level, EAD at onset, calibrated PD):")
el_sensitivity

Account-level link coverage: 25.3% of clean card accounts matched a scored borrower
(scorecard test split covers only part of the book -> account-level but illustrative)

EL across the LGD range (account-level, EAD at onset, calibrated PD):


,scenario,LGD,mean_EL_per_account,segment_EL_total
0,benchmark low,0.65,268.45,93419.0
1,base,0.75,309.74,107791.0
2,downturn,0.85,351.04,122163.0
3,benchmark high,0.90,371.69,129349.0


### Best-estimate-of-EL for defaulted accounts, and EL vs provisions

- **Defaulted accounts (APS 113 Att D EL para 1):** for accounts *already in default*, EL is **not** a
  mechanical long-run x downturn-LGD product. The standard is the **best estimate of expected loss** given
  current conditions - in practice **specific provisions plus a management overlay**. On a demo dataset with
  no recovery cash-flows we cannot compute this; it is documented as the correct treatment.
- **EL vs provisions:** in a bank the EL above would be compared against accounting **provisions** (IFRS 9
  ECL); the **shortfall/excess** flows to CET1 capital. Here EL is illustrative, so this is framing, not a number.
- The EL **sensitivity across the LGD range** (table above) is the honest way to present loss given that LGD
  is assumed, not measured.

## Data-quality finding (the real takeaway of this notebook)

Treat this notebook as a **data-quality finding, not an EAD model**. A sanity check on the
exposure numbers showed that the default flag itself is not trustworthy on this dataset.

- **The 90+ days-past-due flag here does not represent economic default.** It is a mechanical
  delinquency counter, not the moment a borrower actually went bad.
- **Evidence (see the diagnostic cell and traced accounts below):**
  - The largest balance in the year before "default" lands, on average, **~10 months earlier**,
    in a month when the account was **fully current (DPD = 0)** - normal revolving usage, not a run-up into default.
  - The 90-day flag then fires **months later on a tiny stranded residual** (often a few units of currency).
  - Some accounts **revive after their "default" month**, with balances climbing back to ~100k-200k for
    years afterwards - so the flagged month was never a real default at all.
- **Cause:** `SK_DPD` (days past due) **accumulates and never resets**. Once an account is briefly late, the
  counter keeps ticking up on whatever residual remains until it crosses 90 - regardless of whether the borrower
  has actually defaulted.
- **Conclusion:** this dataset **cannot support a credible revolving EAD / CCF model**. The fix already applied -
  anchoring exposure at the **onset of delinquency** (EAD-3) - is the right methodology, but it cannot rescue a
  flag that is itself broken; a **clean, non-reviving default definition** is needed.
- **Where EAD is done properly:** exposure-at-default is demonstrated correctly in the companion
  **Freddie Mac mortgage project**, where the default definition is clean (terminations/defaults are observed
  unambiguously and do not revive).

The EAD/CCF/EL numbers above are kept only as a **methodology demonstration** of the mechanics
(`EL = PD x LGD x EAD`); they are **not credible loss estimates** for this book.

In [11]:
# EVIDENCE (diagnostic only - NOT used as an EAD metric):
# where does the largest balance in the 12 months up to default actually sit?
win = cc.merge(acc[["SK_ID_PREV", "def_month"]], on="SK_ID_PREV")
win = win[(win.MONTHS_BALANCE >= win.def_month - 12) & (win.MONTHS_BALANCE <= win.def_month)]
pk_idx = win.groupby("SK_ID_PREV")["AMT_BALANCE"].idxmax()
pk = (win.loc[pk_idx, ["SK_ID_PREV", "MONTHS_BALANCE", "AMT_BALANCE"]]
        .rename(columns={"MONTHS_BALANCE": "peak_month", "AMT_BALANCE": "peak_bal"}))
ev = acc.merge(pk, on="SK_ID_PREV")

evidence = pd.DataFrame([
    {"check": "peak balance lands IN the default month",  "value": "{:.1%}".format((ev.peak_month == ev.def_month).mean())},
    {"check": "peak balance lands AFTER default (leak)",  "value": str(int((ev.peak_month > ev.def_month).sum()))},
    {"check": "avg months peak sits BEFORE default",      "value": "{:.1f}".format((ev.def_month - ev.peak_month).mean())},
    {"check": "avg peak balance (a healthy-period high)", "value": "{:,.0f}".format(ev.peak_bal.mean())},
    {"check": "avg balance at 90+DPD (secondary EAD)",    "value": "{:,.0f}".format(ev.EAD_90dpd.mean())},
    {"check": "median peak / balance-at-90DPD ratio",     "value": "{:.0f}x".format((ev.peak_bal / ev.EAD_90dpd.replace(0, np.nan)).median())},
])
print("The 12-month peak balance is NOT exposure at default: it sits ~10 months earlier, in a")
print("fully-current period (DPD = 0). Shown here only as evidence of the data-quality problem.\n")
print(evidence.to_string(index=False))


# Two traced accounts: month-by-month balance / limit / days-past-due paths.
def trace(pid, note):
    sel = ev[ev.SK_ID_PREV == pid]
    if sel.empty:
        print(f"\nAccount {pid} - not in the defaulted sample on this run."); return
    r = sel.iloc[0]
    path = cc[cc.SK_ID_PREV == pid].sort_values("MONTHS_BALANCE")
    path = path[path.MONTHS_BALANCE >= int(r.def_month) - 13]
    print(f"\nAccount {pid} - {note}")
    print(f"  def_month={int(r.def_month)}  peak_month={int(r.peak_month)}  "
          f"balance@90DPD={r.EAD_90dpd:,.0f}  peak_balance={r.peak_bal:,.0f}")
    print(f"  {'month':>6}{'balance':>13}{'limit':>11}{'DPD':>6}   note")
    for _, mm in path.iterrows():
        tag = []
        if mm.MONTHS_BALANCE == int(r.peak_month): tag.append("<- PEAK (DPD=0, healthy)")
        if mm.MONTHS_BALANCE == int(r.def_month):  tag.append("<- 90+DPD flag")
        if mm.MONTHS_BALANCE > int(r.def_month):   tag.append("post-flag")
        print(f"  {int(mm.MONTHS_BALANCE):>6}{mm.AMT_BALANCE:>13,.0f}{mm.AMT_CREDIT_LIMIT_ACTUAL:>11,.0f}{int(mm.SK_DPD):>6}   {' '.join(tag)}")

trace(1126779, "peaks while fully current, then 'defaults' months later on a small residual")
trace(2235755, "REVIVES after its 'default' month - balances climb back up for years afterwards")

The 12-month peak balance is NOT exposure at default: it sits ~10 months earlier, in a
fully-current period (DPD = 0). Shown here only as evidence of the data-quality problem.

                                   check  value
 peak balance lands IN the default month   2.8%
 peak balance lands AFTER default (leak)      0
     avg months peak sits BEFORE default   10.6
avg peak balance (a healthy-period high) 59,192
   avg balance at 90+DPD (secondary EAD)  2,964
    median peak / balance-at-90DPD ratio   190x

Account 1126779 - peaks while fully current, then 'defaults' months later on a small residual
  def_month=-2  peak_month=-9  balance@90DPD=9  peak_balance=477,971
   month      balance      limit   DPD   note
     -15      185,756    450,000     0   
     -14      273,331    450,000     0   
     -13      346,209    450,000     0   
     -12      374,650    450,000     0   
     -11      461,407    450,000     0   
     -10      425,124    450,000     0   
      -9      477,971    

In [12]:
# One clean results table (methodology-demonstration figures - see the data-quality finding above).
summary = pd.DataFrame([
    {"metric": "defaulted_accounts_90dpd",        "value": round(len(dmonth), 0)},
    {"metric": "accounts_with_reference",         "value": round(len(acc), 0)},
    {"metric": "accounts_after_homogeneity_excl", "value": round(len(clean), 0)},
    {"metric": "EAD_onset_primary_avg",           "value": round(float(acc.EAD_onset.mean()), 2)},
    {"metric": "EAD_90dpd_secondary_avg",         "value": round(float(acc.EAD_90dpd.mean()), 2)},
    {"metric": "long_run_CCF_ULF_countwt",        "value": round(lr_ccf, 4)},
    {"metric": "downturn_CCF",                    "value": round(dt_ccf, 4)},
    {"metric": "MoC_addon_CCF",                   "value": round(moc_ccf, 4)},
    {"metric": "LGD_base",                        "value": LGD_BASE},
    {"metric": "LGD_downturn",                    "value": LGD_DOWNTURN},
    {"metric": "EL_link_coverage",                "value": round(coverage, 4)},
    {"metric": "mean_EL_per_account_base",        "value": round(float((linked.pd_cal[m]*LGD_BASE*linked.EAD_onset[m]).mean()), 2)},
    {"metric": "mean_EL_per_account_downturn",    "value": round(float((linked.pd_cal[m]*LGD_DOWNTURN*linked.EAD_onset[m]).mean()), 2)},
])
summary.to_csv(OUT_DIR / "ead_summary.csv", index=False)
print("Saved -> outputs/tables/ead_summary.csv  (methodology-demonstration figures, not credible loss estimates)")
summary

Saved -> outputs/tables/ead_summary.csv  (methodology-demonstration figures, not credible loss estimates)


,metric,value
0,defaulted_accounts_90dpd,1806.0000
1,accounts_with_reference,1539.0000
2,accounts_after_homogeneity_excl,1376.0000
3,EAD_onset_primary_avg,2005.0000
4,EAD_90dpd_secondary_avg,2964.3200
5,long_run_CCF_ULF_countwt,-0.8829
6,downturn_CCF,-0.2779
7,MoC_addon_CCF,0.0883
8,LGD_base,0.7500
9,LGD_downturn,0.8500
